# 🧠🤖 第1周·Day 6 ⚡ 代码实战

> **用 NumPy 从零实现 Self-Attention**
>
> 今天的目标：亲手写一遍 Self-Attention 的每一步，真正理解 Q/K/V 计算过程。

## 🎯 任务要求
- 用纯 NumPy，不依赖任何深度学习框架
- 实现 Q/K/V 生成 → 注意力分数 → 缩放 → Softmax → 加权求和
- 完成后验证：每行注意力权重之和 = 1，手动计算 = 矩阵计算

## 💡 提示
- 不会写的地方用 Trae 补全
- 跑通就是胜利！

## Step 1：准备输入矩阵

📌 模拟句子 "我 爱 AI 学习"，4个词，每个词8维向量

In [1]:
import numpy as np

np.random.seed(42)
seq_len = 4
d_model = 8

# 初始化输入矩阵 X, shape = (4, 8)
X = np.random.randn(seq_len, d_model)

print(f"X shape 应该是 (4, 8)，实际: {X.shape if X is not None else '未完成'}")


X shape 应该是 (4, 8)，实际: (4, 8)


## Step 2：生成 Q, K, V

📌 用矩阵乘法 X @ W 得到 Q, K, V，每个 shape 应该是 (4, 4)

In [2]:
d_k = 4

# 生成权重矩阵, shape = (d_model, d_k) = (8, 4)
W_q = np.random.randn(d_model, d_k)
W_k = np.random.randn(d_model, d_k)
W_v = np.random.randn(d_model, d_k)

# 矩阵乘法得到 Q, K, V
Q = X @ W_q
K = X @ W_k
V = X @ W_v

if Q is not None:
    print(f"Q shape: {Q.shape}, K shape: {K.shape}, V shape: {V.shape}")



Q shape: (4, 4), K shape: (4, 4), V shape: (4, 4)


## Step 3：计算注意力分数

📌 Q × K^T 得到注意力分数矩阵

In [3]:
# 计算 Q 和 K^T 的点积
scores = Q @ K.T

if scores is not None:
    print("原始注意力分数:")
    print(np.round(scores, 3))



原始注意力分数:
[[ -1.062  -7.85    1.739   1.609]
 [  5.979  12.407 -18.665  11.327]
 [ -9.143   7.969  11.472  -2.239]
 [ 12.788  -4.557 -21.706   2.642]]


## Step 4：缩放 + Softmax

📌 除以 √d_k 后做 softmax

In [4]:
def softmax(x):
    x_shifted = x - x.max(axis=1, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / exp_x.sum(axis=1, keepdims=True)

# 缩放分数 (除以 sqrt(d_k))
scaled_scores = scores / np.sqrt(d_k)

# 对缩放后的分数做 softmax
attn_weights = softmax(scaled_scores)

if attn_weights is not None:
    print("注意力权重:")
    print(np.round(attn_weights, 3))
    print(f"\n每行求和: {np.round(attn_weights.sum(axis=1), 5)}")
    print("✅ 应该每行都等于 1.0")


注意力权重:
[[0.112 0.004 0.456 0.428]
 [0.025 0.616 0.    0.359]
 [0.    0.148 0.851 0.001]
 [0.994 0.    0.    0.006]]

每行求和: [1. 1. 1. 1.]
✅ 应该每行都等于 1.0


## Step 5：加权求和得到输出

📌 注意力权重 × V = 最终输出

In [5]:
# 用注意力权重对 V 加权求和
output = attn_weights @ V

if output is not None:
    print("输出 shape:", output.shape)
    print("输出:\n", np.round(output, 3))


输出 shape: (4, 4)
输出:
 [[-0.266  1.247 -1.711 -0.26 ]
 [-1.326  2.649 -4.731  0.253]
 [-1.782  0.425  0.94   0.866]
 [ 2.312  0.294 -0.452 -2.815]]


## Step 6：验证

📌 手动计算第0个词的输出，对比矩阵结果

In [6]:
if attn_weights is not None and V is not None:
    word0_weights = attn_weights[0]
    word0_output = word0_weights @ V

    print("手动计算第0词输出:", np.round(word0_output, 4))
    print("矩阵计算第0词输出:", np.round(output[0], 4))
    print("是否一致:", np.allclose(word0_output, output[0]))
    print("\n✅ 如果一致，说明你的实现是正确的！")

    # 打印注意力权重热力图
    words = ["我", "爱", "AI", "学习"]
    print("\n注意力权重热力图:")
    print("         " + "  ".join(f"{w:^6}" for w in words))
    for i, w in enumerate(words):
        row = "  ".join(f"{attn_weights[i][j]:.3f}" for j in range(seq_len))
        print(f"{w:^6}  {row}")


手动计算第0词输出: [-0.2656  1.247  -1.7113 -0.2598]
矩阵计算第0词输出: [-0.2656  1.247  -1.7113 -0.2598]
是否一致: True

✅ 如果一致，说明你的实现是正确的！

注意力权重热力图:
           我       爱       AI      学习  
  我     0.112  0.004  0.456  0.428
  爱     0.025  0.616  0.000  0.359
  AI    0.000  0.148  0.851  0.001
  学习    0.994  0.000  0.000  0.006


## 🎉 恭喜完成！

你刚刚用纯 NumPy 实现了一个完整的 Self-Attention！

回顾关键步骤：
1. **输入 X** → 2. **生成 Q/K/V** → 3. **Q·K^T 分数** → 4. **缩放 + Softmax** → 5. **加权求和**

Day 2 学的多头注意力，就是多个这样的过程并行，最后拼接。

> 💡 **进度: W1 Day 6/7 | ⚡实战日**